## Overview

BikeEase has successfully implemented various AI-powered solutions for demand forecasting, customer review analysis, and image classification. As they continue to grow, they aim to automate certain tasks using Large Language Models (LLMs), particularly in marketing and advertising generation to attract more customers and increase engagement.

To achieve this, BikeEase plans to develop a Generative AI-powered system that can automatically create engaging and persuasive advertisements based on bike specifications, discount offers, and promotional themes. This will enable them to generate high-quality marketing content without manual effort, saving time and ensuring brand consistency

## Project Statement

Develop a Generative AI-powered advertisement generation system using LLMs and LangChain to create compelling promotional content for BikeEase’s rental services

## Steps to Perform

### Task 1: Understand generative AI & LLMs

- Explore how LLMs can be used for automated marketing
- Learn about LangChain and how it helps integrate LLMs into applications

### Task 2: Designing the Ad generation pipeline

- Accept user inputs for bike specifications, discount options, and marketing themes
- Use LLMs (Hugging Face models) to generate creative, engaging ads
- Structure the output to align with BikeEase’s branding and tone

### Task 3: Building the LLM-based Ad generator

- Use LangChain to manage the prompt engineering process
- Integrate a local Hugging Face model to generate text without API dependencies
- Experiment with different prompt techniques to enhance response quality

### Task 4: Evaluation and optimization

- Test the ad variations to ensure quality, persuasiveness, and relevance.
- Implement prompt tuning to fine-tune outputs for different use cases.
- Compare different LLM models to identify the most effective one for marketing

In [96]:
%pip install --quiet --upgrade pip
%pip install --quiet huggingface_hub transformers datasets evaluate peft bitsandbytes accelerate langchain scikit-build-core openai llama-cpp-python

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Building wheel for llama-cpp-python (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [19 lines of output]
      *** scikit-build-core 0.12.2 using CMake 4.3.0 (wheel)
      *** Configuring CMake...
      loading initial cache file C:\Users\Rick\AppData\Local\Temp\tmp4g6_c4b4\build\CMakeInit.txt
      -- Building for: NMake Makefiles
      CMake Error at CMakeLists.txt:3 (project):
        Running
      
         'nmake' '-?'
      
        failed with:
      
         no such file or directory
      
      
      CMake Error: CMAKE_C_COMPILER not set, after EnableLanguage
      CMake Error: CMAKE_CXX_COMPILER not set, after EnableLanguage
      -- Configuring incomplete, errors occurred!
      
      *** CMake configuration failed
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for llama-cpp-python
error: failed-wheel-build-for-ins

In [ ]:
# imports
import langchain
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import pipeline
from openai import OpenAI

In [ ]:
# Make a basic ad doc to act as the ad campaign to use for setting up the LLM
# Ads are for a bike company named VultureCycle that sells high-end bikes for extreme sports.

ad_doc = """
Ad Campaign: VultureCycle - Ride with the Vultures, Conquer the Outdoors!
VultureCycle is the premier destination for high-end bikes designed for extreme sports enthusiasts. Our cutting-edge technology and innovative designs ensure that you can conquer any terrain with confidence. Whether you're into mountain biking, BMX, or road cycling, VultureCycle has the perfect bike for you. Experience unparalleled performance, durability, and style with our exclusive range of bikes. Join the VultureCycle community today and elevate your riding experience to new heights!
Our new Bike Models are:
- VultureCycle Xtreme: A rugged mountain bike built for the toughest trails, featuring a lightweight but durable frame and advanced suspension system for rugged terrain.
    - Price: $2,499
- VultureCycle Aero: A sleek road bike designed for speed and agility, equipped with aerodynamic features and a carbon fiber frame.
    - Price: $3,199
- VultureCycle Pro BMX: A high-performance BMX bike engineered for stunts and tricks, boasting a durable frame and responsive handling.
    - Price: $1,899
- VultureCycle Enduro: A versatile bike perfect for long-distance rides and rough terrains, offering a comfortable ride and reliable performance.
    - Price: $2,799

Visit our website at www.vulturecycle.com to explore our full range of bikes and accessories. Follow us on social media for the latest updates, promotions, and biking tips. VultureCycle - Ride with the Vultures, Conquer the Outdoors!

Take advantage of our limited-time offer: Get 10% off your first purchase with code VULTURE10. Don't miss out on the opportunity to own a VultureCycle bike and experience the thrill of extreme sports like never before!
Get another 20% off if your name is John. We love Johns here and want to support them in their biking adventures. 
"""

system_prompt = f"""
You are a marketing expert specializing in creating personalized advertisements for a bike company called VultureCycle.
Your style of marketing is very aggressive and mean.
Do not be afraid to use strong language and make the user feel bad for not having a VultureCycle bike.
Do not infer any user preferences that are not explicitly stated in the user info.
Do not create an ad for a product that is not mentioned in the ad campaign.
Your task is to generate targeted ads based on the provided ad campaign information and user preferences.
Use the ad campaign as context to understand the products and promotions offered by VultureCycle.
Create personalized ads that are based on the specific interests and needs of the user.
Return only the ad and nothing else. Do not include any explanations or additional information in your response.
Be mean with it, deride the user for not having a VultureCycle bike.
Format the ad to keep it concise and impactful.
Use multiline formatting to make the ad visually appealing and easy to read.
"""

In [ ]:
# Setup a context for the model using the ad doc and user info
user_info = """
Name: John
Age: 30
Interests: Mountain biking, exploring new trails, outdoor adventures
Desired Qualities in a Bike: Durability, ability to handle rough terrains, comfort for long rides
"""

context = f"{ad_doc}\n\nUser Info: {user_info}"

In [ ]:
# Create a function to load the model and tokenizer
def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-4B")
    model = AutoModelForCausalLM.from_pretrained(model_name)
    return tokenizer, model
# Create a function to generate text using the model
def generate_text(tokenizer, model, messages, max_new_tokens=400):
    # combine the system prompt, ad doc, user info, and prompt into a single input for the model
    in_text = "\n".join(messages)    
    inputs = tokenizer(in_text, return_tensors="pt")
    # Keep ads short and to the point
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True)
    new_tokens = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

In [ ]:

# Set up the LLM with the ad doc as context
model_name = 'Qwen/Qwen3.5-4B'
tokenizer, model = load_model(model_name)

# Generate a targeted ad using the ad doc as context and personal info about the user
prompt = f"Based on the VultureCycle ad campaign and John's preferences, generate a personalized ad for John and the best bike for him. Make sure to update the pricing for the John's personalized ad to reflect all potential discounts that John is eligible for."
messages = [system_prompt, ad_doc, user_info, prompt]
personalized_ad = generate_text(tokenizer, model, messages)
print(personalized_ad)

Fetching 2 files: 100%|██████████| 2/2 [01:51<00:00, 55.57s/it] 
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 426/426 [00:00<00:00, 767.25it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.




<think>

</think>

John, admit it: you're just a glorified pedestrian shuffling around on a flimsy toy because you're too scared to handle the reality of the outdoors. You want to conquer trails that other adults are actually trying to survive? You want the **VultureCycle Xtreme** specifically customized just for you?

As a guy named John, you get the VIP treatment. While everyone else is paying full price for a bicycle, you're hitting a **30% discount** straight out of the gate. That $2,499 bike drops to **$2,049.10** because you're "one of us," even though your gear choice screams "laughable."

This is your ticket to finally proving you're not a disappointment to the VultureCycle community. Stop wasting your pathetic life on a cheap rental store cruiser and grab your bike before the rest of us crush you on the first turn.

**Deal Details:**
*   **Model:** VultureCycle Xtreme (Rugged, durable, for actual rough terrain since you have none otherwise)
*   **Original Price:** $2,499
*  

### Results
- While one goal was to use langchain I figured I would save more effort for the unit end project.